# Qualifire — On-demand HTML dashboards

Read the **system table** (last N days) and render the static, health,
and interactive HTML dashboards programmatically. Pick the backend at
the top of the notebook so the same cells work on:

| # | Environment | `BACKEND` | `SYSTEM_TABLE` |
|---|---|---|---|
| 1 | Local dev | `'sqlite'` | path to a `.sqlite` file |
| 2 | AIDP Workbench (Oracle ADW via external catalog) | `'external_catalog'` | `catalog.schema.qualifire_history` |
| 3 | AIDP Workbench (Delta Lake) | `'delta'` | fully-qualified Delta table |
| 4 | Generic JDBC datasource (caller supplies driver + URL) | `'jdbc'` | remote DB table name |

> **`external_catalog` vs `jdbc`** — when AIDP exposes ADW (or any
> external store) through its catalog, the qualifire history table is
> a regular catalog-resident table and `'external_catalog'` is the
> right backend. The `'jdbc'` backend is reserved for connecting to
> a generic JDBC-compliant datasource directly — caller supplies the
> driver jar and connection URL.

**Every cell below calls public APIs from `qualifire.reporting`** —
the notebook itself doesn't invent any rendering logic. Treat it as
both a worked example for end users and a smoke test of the API
surface. All three HTML files are written to `OUTPUT_DIR`. Re-run any
single render cell at any time — they each read fresh from the
system table, so the dashboards always reflect the latest persisted
state.

## 1. Configure the backend + output directory

**Edit the next cell directly.** AIDP Workbench has no terminal, so
set the constants in-cell — the env vars (`QF_BACKEND`,
`QF_SYSTEM_TABLE`, …) are a secondary path for shells that have one.

Constants you'll touch:

- `BACKEND` — `'sqlite'` (local), `'external_catalog'` (AIDP / Hive
  / Unity Catalog / Oracle ADW), `'delta'`, or `'jdbc'`.
- `SYSTEM_TABLE` — path (sqlite) or fully-qualified table name (others).
- `OUTPUT_DIR` — where to write generated HTML files. On AIDP prefer
  a project-mounted directory so the files survive kernel restarts.
- `DAYS` — lookback window for the report; the interactive
  dashboard's in-page picker can extend this further.
- `JDBC_CONFIG` — fill in URL + driver + credentials when
  `BACKEND='jdbc'`.

In [ ]:
import os, tempfile
from pathlib import Path

# ---------------------------------------------------------------------------
# Edit the constants in THIS CELL to match your environment.
# (Env vars work too, but they're a fallback — AIDP Workbench has no
# terminal, so editing the cell is the primary path.)
# ---------------------------------------------------------------------------

# Pick one of: 'sqlite' | 'external_catalog' | 'delta' | 'jdbc'
#
#   Local dev:
#       BACKEND = 'sqlite'
#   AIDP + Oracle ADW (external catalog) or any catalog-resident
#   table (Hive / Unity Catalog):
#       BACKEND = 'external_catalog'
#   Delta Lake:
#       BACKEND = 'delta'
#   Generic JDBC datasource (caller supplies driver + URL):
#       BACKEND = 'jdbc'
BACKEND = os.environ.get('QF_BACKEND', 'sqlite')

# Where the system table lives — interpretation depends on BACKEND.
#   sqlite           → filesystem path to .sqlite file
#                      e.g. '/Users/me/qualifire/qualifire_history.sqlite'
#   external_catalog → 'catalog.schema.table'
#                      e.g. 'aidp_catalog.qualifire.history'
#   delta            → 'catalog.schema.table' or 'delta.`/path/to/table`'
#   jdbc             → table name in the remote DB
#                      e.g. 'QF_USER.QUALIFIRE_HISTORY'
SYSTEM_TABLE = os.environ.get(
    'QF_SYSTEM_TABLE',
    str(Path(tempfile.gettempdir()) / 'qualifire_history.sqlite'),
)

# Where to drop the generated HTML files. Use a path that's
# persistent on AIDP (the working directory or a project-mounted
# folder), not /tmp, unless you only want to view inline once.
OUTPUT_DIR = Path(os.environ.get('QF_OUTPUT_DIR', tempfile.mkdtemp(prefix='qf_dashboards_')))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Default lookback. The interactive dashboard's in-page picker can
# extend this further; the static + health reports fix it at gen-time.
DAYS = int(os.environ.get('QF_DAYS', '90'))

# Required only when BACKEND == 'jdbc'. Generic JDBC values:
#
#   url:      'jdbc:oracle:thin:@//<host>:<port>/<service_name>'
#             or the TNS form 'jdbc:oracle:thin:@<tns_alias>?TNS_ADMIN=/path/to/wallet'
#   user:     wallet user (or DB user)
#   password: wallet password (or DB password)
#   driver:   'oracle.jdbc.OracleDriver' for Oracle, vendor-specific otherwise
#
# For a wallet/cloud-config-based connection, set TNS_ADMIN in the URL
# fragment instead of plain user/password.
JDBC_CONFIG = {
    'url':      os.environ.get('QF_JDBC_URL', ''),
    'user':     os.environ.get('QF_JDBC_USER', ''),
    'password': os.environ.get('QF_JDBC_PASSWORD', ''),
    'driver':   os.environ.get('QF_JDBC_DRIVER', 'oracle.jdbc.OracleDriver'),
}

print('BACKEND      :', BACKEND)
print('SYSTEM_TABLE :', SYSTEM_TABLE)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('DAYS         :', DAYS)
if BACKEND == 'jdbc':
    print('JDBC URL     :', JDBC_CONFIG['url'] or '(not set — fill in JDBC_CONFIG above)')

## 2. Open the system-table storage handle

[`qualifire.reporting.make_storage`](../../qualifire/reporting/storage_factory.py)
is the public factory: it returns a SystemTableStorage for the
configured backend. JDBC / external_catalog / delta need a live
`SparkSession` (PySpark); the factory builds / attaches to a local
one via `getOrCreate` if you don't pass `spark=...`. The sqlite
backend has no Spark dependency.

The notebook stops here if the backend's prerequisites aren't met,
rather than failing deep inside a render cell.

In [ ]:
from collections import Counter

from qualifire.reporting import make_storage

storage = make_storage(
    BACKEND, SYSTEM_TABLE,
    JDBC_CONFIG if BACKEND == 'jdbc' else None,
)
print('storage :', type(storage).__name__)

# Sanity-check: read a small slice and report the row count by status.
rows = storage.read_health_data(days=DAYS)
print(f'Loaded {len(rows)} validation rows for the last {DAYS} days.')
if rows:
    counts = Counter((r.get('validation_status') or 'UNKNOWN') for r in rows)
    for sev, n in counts.most_common():
        print(f'  {sev:>8} : {n}')

## 3. Health report (static HTML)

Aggregated counters + trend bar-chart. Lightweight, no JS dependency.

In [ ]:
from IPython.display import HTML, display

from qualifire.reporting import HealthReporter, generate_health_html

report = HealthReporter(storage).generate(days=DAYS)
health_path = OUTPUT_DIR / f'health_report_{DAYS}d.html'
generate_health_html(report, output_path=str(health_path))

print(f'Health report : {health_path}  ({health_path.stat().st_size:,} bytes)')
print(
    f'  total checks: {report.total_checks}  '
    f'pass={report.pass_rate}%  warn={report.warning_rate}%  err={report.error_rate}%'
)

display(HTML(health_path.read_text()))

## 4. Interactive dashboard (Plotly)

Drill-down view: pick a dataset → its validations → a single
validation's per-partition history. Time range editable in-page
(default = `DAYS`). Adapts to OS dark mode automatically.

Plotly loads from CDN, so the file size stays small (~20–60 KB
depending on snapshot size). For air-gapped deploys, swap the script
src for a local copy of `plotly.min.js`.

In [ ]:
from qualifire.reporting import generate_interactive_html

interactive_path = OUTPUT_DIR / f'interactive_dashboard_{DAYS}d.html'
generate_interactive_html(storage, output_path=str(interactive_path), days=DAYS)
print(f'Interactive : {interactive_path}  ({interactive_path.stat().st_size:,} bytes)')

display(HTML(interactive_path.read_text()))

## 5. Latest-run summary (static HTML, optional)

`generate_html_report(QualifireResult)` renders **a single run**
rather than a long-window aggregate. Useful right after a pipeline
run; less useful when reading the system table directly. Skip if you
only want the trend / interactive views.

[`qualifire.reporting.build_result_from_system_table`](../../qualifire/reporting/system_table.py)
reconstructs a `QualifireResult` from the persisted rows so you don't
have to reach back to the engine that produced it.

In [ ]:
from qualifire.reporting import (
    build_result_from_system_table,
    generate_html_report,
)

if not rows:
    print('No rows in system table — skip.')
else:
    qr = build_result_from_system_table(storage, days=DAYS)
    static_path = OUTPUT_DIR / f'latest_run_{(qr.run_id or "unknown")[:8]}.html'
    generate_html_report(qr, output_path=str(static_path))
    print(f'Per-run report : {static_path}')
    display(HTML(static_path.read_text()))

## 6. Files written

All three HTML files live in `OUTPUT_DIR`. They're self-contained —
open them in any browser, attach to email, drop on a static-file
host, etc.

In [ ]:
for p in sorted(OUTPUT_DIR.glob('*.html')):
    size = p.stat().st_size
    print(f'  {p}  ({size:,} bytes)')